Pulls data from API, filters for taxon_id 1280 = Staphylococcus aureus that was identified as resistant via wet lab. Then pull out datapoints that show resistance or susceptibility specifically to methicillin.

In [1]:
import requests
import pandas as pd
from io import StringIO

url = "https://www.bv-brc.org/api/genome_amr/?eq(taxon_id,1280)&eq(evidence,Laboratory%20Method)&limit(50000)"
headers = {"Accept": "text/csv"}

response = requests.get(url, headers=headers)
df = pd.read_csv(StringIO(response.text))

# Filter to methicillin, drop Intermediate (keep binary classification clean)
methicillin_df = df[df['Antibiotic'] == 'methicillin'].copy()
methicillin_df = methicillin_df[methicillin_df['Resistant Phenotype'].isin(['Resistant', 'Susceptible'])]

print("Shape:", methicillin_df.shape)
print("\nClass balance:")
print(methicillin_df['Resistant Phenotype'].value_counts())
print("\nSample genome IDs:")
print(methicillin_df['Genome ID'].head(10).tolist())

# Save it
methicillin_df.to_csv("methicillin_labels.csv", index=False)
print("\nSaved methicillin_labels.csv")

Shape: (1082, 21)

Class balance:
Resistant Phenotype
Susceptible    574
Resistant      508
Name: count, dtype: int64

Sample genome IDs:
[1280.4727, 1280.8343, 1280.2483, 1280.8315, 1280.7953, 1280.25081, 1280.24873, 1280.2511, 1280.8168, 1280.9764]

Saved methicillin_labels.csv


In [2]:
methicillin_df.head()

,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
17,1280,1280.4727,Staphylococcus aureus strain USFL197,methicillin,Resistant,NaN,NaN,NaN,NaN,Disk diffusion,...,NaN,NaN,CLSI,2013.0,NaN,NaN,NaN,Laboratory Method,NaN,29487865.0
20,1280,1280.8343,Staphylococcus aureus C00013400,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0
142,1280,1280.2483,Staphylococcus aureus strain 12483_8_66,methicillin,Resistant,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,British Society for Antimicrobial Chemotherapy...,NaN,NaN,NaN,NaN,Laboratory Method,NaN,30696529.0
143,1280,1280.8315,Staphylococcus aureus C00013372,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0
152,1280,1280.7953,Staphylococcus aureus C00001155,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0


Hitting limit of 25,000 results

In [58]:
import requests

genome_list = methicillin_df['Genome ID'].tolist()
genome_string = ",".join([str(x) for x in genome_list])

response = requests.post(
    "https://www.bv-brc.org/api/sp_gene/",
    headers={"Accept": "text/csv", "Content-Type": "application/rqlquery+x-www-form-urlencoded"},
    data=f'in(genome_id,({genome_string}))&eq(property,%22Antibiotic%20Resistance%22)&limit(500000)'
)

genes_df = pd.read_csv(StringIO(response.text))

print("Shape:", genes_df.shape)
print("\nUnique genomes with gene data:", genes_df['Genome ID'].nunique())
print("\nTop 15 most common genes:")
print(genes_df['Gene'].value_counts().head(15))

Shape: (25000, 21)

Unique genomes with gene data: 840

Top 15 most common genes:
Gene
GdpD         354
gyrA         253
rpoB         246
MurA         243
gyrB         241
rpoC         239
mprF         146
folA, Dfr    139
S10p         138
blaZ         136
BceB         134
rho          133
mepA         132
Alr          131
mgrA         130
Name: count, dtype: int64
